# RLHF (Reinforcement Learning from Human Feedback)

**Tags:** finetuning, alignment
**Prerequisites:** Fine-tuning
**Related Concepts:** See flowchart below
**Source:** llm/concepts/rlhf.md

## TL;DR

Three-stage pipeline: (1) SFT on instruction-response pairs, (2) collect human preference pairs, train reward model to predict preferences, (3) use PPO to fine-tune policy against reward model. Aligns LLM with human values. Trade-off: complex, expensive, but foundational technique for ChatGPT/Claude alignment.

## Core Intuition

Humans can't exhaustively specify what makes a "good" response. But they can compare two outputs: "A is better than B." Collect thousands of such comparisons, learn patterns (reward model), then optimize the LLM to maximize that reward. This is how ChatGPT/Claude are aligned.

## How It Works

**Stage 1: Supervised Fine-Tuning (SFT)**
```
Goal: Establish baseline instruction-following

Data: Pairs of (prompt, response) curated by humans
Training: Standard language modeling on response tokens

Example:
Prompt: "Explain machine learning"
Response: "Machine learning is a subset of AI where..."

Objective: minimize -log P(response | prompt)

Output: Base model π_SFT that can follow instructions reasonably well
Compute: 1-2 weeks on A100
```

**Stage 2: Reward Model Training**
```
Goal: Learn to predict human preferences

Data collection:
1. Generate multiple outputs per prompt using π_SFT
2. Show pairs (A, B) to human annotators
3. Collect preference judgment: "A is better" or "B is better"
4. Result: dataset of (prompt, preferred_output, dispreferred_output)

Typical scale: 10k-50k comparisons from 1k-5k prompts

Reward model training:
- Use language model (often smaller, e.g., 350M)
- Binary classification: given (prompt, response), predict human preference
- Loss: minimize -log σ(r(prompt, preferred) - r(prompt, dispreferred))
  where σ is logistic function

Output: Reward function r(prompt, response) → scalar [-∞, +∞]
  - High scores: human-preferred responses
  - Low scores: undesired responses

Cost: days to a week of training
```

**Stage 3: Reinforcement Learning (PPO)**
```
Goal: Maximize reward while staying close to SFT model

Algorithm: Proximal Policy Optimization (PPO)

Objective function:
L_RL = E[r(x, y)] - β * KL(π_RL || π_SFT)

Where:
- r(x, y): reward from reward model
- π_RL: policy being optimized
- π_SFT: reference/base policy
- β: KL penalty coefficient (controls divergence)

Training loop (per prompt):
1. Sample response y from π_RL (current policy)
2. Score response using reward model: r(x, y)
3. Compute policy gradient: ∇ log π_RL(y|x) * (r(x, y) - baseline)
4. Apply PPO clipping to limit policy change per step
5. Update π_RL

PPO clipping (prevents too-large updates):
  L_clip = E[min(ratio * advantage, clip(ratio, 1-ε, 1+ε) * advantage)]
  where ratio = π_RL(y|x) / π_SFT(y|x), ε ≈ 0.2

Cost: weeks on A100 for large models
```

**Visual Pipeline:**
```
Human preferences (comparisons)
        ↓
   Reward Model Training
        ↓
    r(x, y) reward function
        ↓
   PPO Fine-tuning
        ↓
   Aligned Policy π_RL
```

### Workflow Diagram**Note:** This is a basic workflow template. Review and customize based on specific concept.
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

## Key Properties & Trade-offs

/ Trade-offs

| Stage | Data Needed | Duration | Cost | Compute |
|-------|------------|----------|------|---------|
| SFT | 10k-100k (x, y) pairs | 1-2 weeks | $ | 1-2 A100 weeks |
| Reward Model | 10k-50k (x, y_pref, y_dis) | 3-7 days | $ | 1 A100 week |
| PPO | Same as RM data | 2-4 weeks | $$$ | 2-4 A100 weeks |
| **Total** | - | **4-7 weeks** | **$$$** | **4-7 A100 weeks** |

**RLHF vs Alternatives:**

| Method | Quality | Speed | Cost | Complexity |
|--------|---------|-------|------|-----------|
| SFT only | 70-80% | Fast | $ | Low |
| RLHF | 90-95% | Slow | $$$ | High |
| DPO | 90-94% | Medium | $ | Medium |
| PPO-based | 92-96% | Slow | $$$ | Very High |

### Code Implementation

```python
# Key imports for RLHF (Reinforcement Learning from Human Feedback)
import numpy as np
import torch
from typing import Any

# RLHF (Reinforcement Learning from Human Feedback) example implementation
class RLHF(ReinforcementLearningfromHumanFeedback):
    """
    RLHF (Reinforcement Learning from Human Feedback) implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

### Common Interview Questions

**Q: What is RLHF (Reinforcement Learning from Human Feedback) used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of RLHF (Reinforcement Learning from Human Feedback)?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does RLHF (Reinforcement Learning from Human Feedback) compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using RLHF (Reinforcement Learning from Human Feedback)?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind RLHF (Reinforcement Learning from Human Feedback)?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/rlhf.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]